In [252]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [258]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other-1)) * out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):  # other + self
        return self + other

    def __rsub__(self, other):      # other - self
        return other + (-self)

    def __rtruediv__(self, other):  # other / self
        return other * self**-1

    def __truediv__(self, other):
        return self * (other**-1)

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    
    def backward(self):

        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()





    

L = d * f
dL/dd = f
(f(x+h)-f(x))/h
((d+h)*f - d*f)/h
(d*f + h*f - d*f)/h
hf/h
=f=-2




crux:
dL/dc = ?
d = e+c
ð/ðc= 1 + 0 'partial derivation'
dd/dc = 1.0
dd/de= 1.0



using chain rule:
dL/dc = dd/dc * dL/dd
dL/dc = -2 * 1
dL/dc = -2
dL/de = -2


dL/da = dL/de * de/da
de/da= ((f(x+h)-f(x))/h)*-2
(((a+h)*b - ab)/h)*-2
((ab - bh - ab)/h)*-2
(bh/h)*-2
=-2b= -2 * -3 = 6.0

dL/db = a*dL/de=-4.0

In [ ]:


'''L derivative in resoect to L is 0.999999999... (floating point error) -> 1.0
L.grad = 1.0
f.grad = 4.0
d.grad = -2.0
c.grad = -2
b.grad = -4
a.grad = 6


a.data += 0.01 * a.grad
b.data += 0.01 * b.grad
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad

e = a*b
d = e+c
L = d*f

print(L.data)

'''

-1.536000000000001


In [254]:
'''inputs x1 x2'''

x1 = Value (2.0, label='x1')
x2 = Value (0.0, label='x2')

'''weights w1 w2'''
w1 = Value (-3.0, label='w1')
w2 = Value(1.0, label='w2')
'''bias of neuron'''
b = Value(6.8813735870195432, label='b')

# x1*w1 + x2*w2 +b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2. label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'



e = (2*n).exp()
o = (e-1)/(e+1) 
o.label = 'o'
o.backward()





In [421]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
      
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x= layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]





do/do = 1.0
do/dn = 1 - tanh(n)**2
1 - o**2
1 - 0.7071067811865476**2 
=0.5

Value(data=-0.14535608865820787)

In [466]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]


In [475]:
for k in range(1):
    #forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    #zerograd and backwardpass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    #update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    #just to check
    print(k, loss.data)



0 0.001733853432557617


In [ ]:
class Loss:
    def calculate(self, output, ys):
        sampleLosses = self.forward(output, ys)
        dataLoss = np.mean(sampleLosses)
        return dataLoss
    
class lossCategoricalCrossEntropy(Loss):
    def forward(self, ypred, yTrue):
        samples = len(ypred)
        yPredClipped = np.clip(ypred, 1e-7, 1 - 1e-7)
        if len(yTrue.shape) == 1:
            correctConfidence = yPredClipped[range(samples), yTrue]
        elif len(yTrue.shape) == 2:
            correctConfidence = np.sum(yPredClipped*yTrue, axis = 1)
        negLoLikelyhood = -np.log(correctConfidence)
        return negLoLikelyhood

[Value(data=0.9816021878175154),
 Value(data=-0.9752251981536647),
 Value(data=-0.9830679365777322),
 Value(data=0.9776126820323223)]

In [ ]:
class activationReLU:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)
        
class activationSoftmax:
    def forward(self, logits):
        z = logits - logits.max(axis = 1, keepdims = True)
        e = np.exp(z)
        self.output = e / e.sum(axis = 1, keepdims = True)